# 배터리 환경 — 정해진 글자 배열 학습 + 바람 robustness (Colab Runner)

**환경**: `BatteryShapeFormationEnv` 한 종류만 사용. 바람 유무는 `--wind-prob/--randomize-wind` 옵션으로 한 스크립트(`comm_train_battery.py`)에서 처리.

**두 정책을 한 노트북에서 학습**:
- **A (battery)**: 배터리 제약만, 바람 없음
- **B (battery + wind robust)**: 배터리 제약 + 바람 도메인 랜덤화 `[0, WIND_PROB]`

**산출물 (모두 Drive 직저장)**:
- ckpts / TensorBoard runs
- GIF (각 정책 + 바람 0 / 바람 EVAL_WIND_PROB)
- eval txt (각 정책)
- 바람 sweep PNG (A vs B 비교 곡선)

**런타임 끊김 대응**: 셀 6의 `latest_ckpt()` 헬퍼가 Drive를 보고 두 정책 각각의 최신 ckpt를 자동 감지. 학습 셀(8/9) 그대로 다시 실행하면 끊긴 지점부터 이어짐.

**런타임**: `런타임 → 런타임 유형 변경 → GPU (T4)` 필수.

**한쪽만 학습하고 싶을 때**: 셀 8(=A) 또는 셀 9(=B) 중 원하는 쪽만 실행하면 됨. 평가/스윕 셀은 양쪽 ckpt가 다 있어야 동작.

## 1. GitHub clone

복구 시에도 이 셀만 다시 돌리면 OK. Drive 저장된 ckpt/log는 그대로 살아있음.

In [4]:
BRANCH = "Saehoon"

# 어떤 %cd 상태에서 다시 돌려도 깨끗이 다시 받도록 절대경로 사용.
# (상대경로 + 이미 CWD가 /content/RL-2026s1-tp 안인 경우 RL-2026s1-tp/RL-2026s1-tp/ 가
# 중첩 생성되는 버그 회피.)
%cd /content
!rm -rf /content/RL-2026s1-tp
!git clone https://github.com/umbrellalily/RL-2026s1-tp.git /content/RL-2026s1-tp
%cd /content/RL-2026s1-tp

!git fetch origin
!git switch $BRANCH
!git pull origin $BRANCH

!git log --oneline -1
!grep -n 'start.iter' comm_train_battery.py | head -3 || echo '⚠️ --start-iter 코드 없음 — 코드 동기화 확인 필요'
!grep -n 'wind_prob' comm_train_battery.py | head -3 || echo '⚠️ wind 인자 없음 — 코드 동기화 확인 필요\'

/content
Cloning into '/content/RL-2026s1-tp'...
remote: Enumerating objects: 219, done.
remote: Counting objects: 100% (219/219), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 219 (delta 80), reused 175 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (219/219), 19.18 MiB | 18.93 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/RL-2026s1-tp
Branch 'Saehoon' set up to track remote branch 'Saehoon' from 'origin'.
Switched to a new branch 'Saehoon'
From https://github.com/umbrellalily/RL-2026s1-tp
 * branch            Saehoon    -> FETCH_HEAD
Already up to date.
566c850 (HEAD -> Saehoon, origin/Saehoon) maxstep 개선 runner
244:        "--start-iter",
250:            "folder with incrementing ckpt numbers (e.g. --start-iter 110 → next save is "
442:        global_iter = args.start_iter + it + 1
56:    wind_prob: float,
78:        wind_prob=wind_prob,
286:        wind_prob=args.wind_prob,


## 2. Google Drive 마운트 + 저장 경로

모든 산출물은 `DRIVE_ROOT` 아래로 직저장. 시퀀스를 바꿔 가며 학습하려면 셀 4의 `EXP_TAG`만 바꾸면 됨 → A/B 둘 다 별도 폴더로 자동 분리.

In [5]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery_letters_no_battery"
!mkdir -p {DRIVE_ROOT}/ckpts {DRIVE_ROOT}/runs {DRIVE_ROOT}/gifs {DRIVE_ROOT}/evals {DRIVE_ROOT}/sweeps
print("DRIVE_ROOT =", DRIVE_ROOT)
!ls -la {DRIVE_ROOT}

Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/drone_results/battery_letters_no_battery
total 20
drwx------ 2 root root 4096 May 29 08:27 ckpts
drwx------ 2 root root 4096 May 29 08:27 evals
drwx------ 2 root root 4096 May 29 08:27 gifs
drwx------ 2 root root 4096 May 29 08:27 runs
drwx------ 2 root root 4096 May 29 08:27 sweeps


## 3. 패키지 설치

In [6]:
!pip install -q torchrl pettingzoo==1.24.3 gymnasium scipy matplotlib pillow tensorboard

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.8/847.8 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 537.6/537.6 kB 50.2 MB/s eta 0:00:00


## 4. 학습 시퀀스 + 환경/학습/바람 설정

**시퀀스 한 번에 하나만 학습** (random sampling 없음). 다른 시퀀스는 `EXP_TAG`를 바꿔서 별도로.

예시: `GROUND→B→A→Y→C` (4단계). 단계가 늘수록 `MAX_STEPS`/`TOTAL_FRAMES` 키우기.

**바람 설정**: `WIND_PROB=0.3` → 정책 B 학습 시 매 에피소드 바람 세기 `[0, 0.3]`·방향 랜덤. 평가용 고정 바람은 `EVAL_WIND_PROB`.

In [7]:
# ============================================================
# 학습/평가 공통 설정
# ============================================================
TARGET_SEQUENCE = "GROUND,D,G"        # ← 학습/평가 시퀀스 (자유 변경)
EXP_TAG         = "DG"                # ← Drive 폴더명에 들어감

# 환경
GRID_SIZE = 25
N_AGENTS  = 14
MAX_STEPS = 500                       # 단계당 ~100 step 권장

# 학습 hyperparam (양쪽 정책 공통)
TOTAL_FRAMES     = 1_200_000
FRAMES_PER_BATCH = 4096
MINIBATCH_SIZE   = 512
PPO_EPOCHS       = 6
LR               = 2e-4
ENT_COEF         = 0.01
CLIP_EPS         = 0.15
CKPT_EVERY       = 5                  # disconnection 시 최대 손실 ~5 iter

# Battery hyperparam (양쪽 정책 공통)
INITIAL_BATTERY          = 1
HOVER_BATTERY_COST       = 0
MOVE_BATTERY_COST        = 0
LOW_BATTERY_MOVE_PENALTY = 0

# Reward shaping (양쪽 정책 공통)
COMPLETION_REWARD       = 50.0
ASSIGNED_TARGET_REWARD  = 0.1
COVERAGE_DELTA_REWARD   = 0.3
HOVER_PENALTY           = 0.05
SHAPING_COEF            = 0.5

# 바람 hyperparam (정책 B 학습 / 평가에서 사용)
WIND_PROB      = 0.3                  # B 학습 시 바람 상한
WIND_STRENGTH  = 1
EVAL_WIND_PROB = 0.3                  # GIF/eval 시 B에 거는 고정 바람
WIND_LEVELS    = "0.0,0.1,0.2,0.3,0.4"  # sweep용

# A→B 전이학습 옵션 (B를 처음부터 학습하지 않고 A의 정책에서 finetune)
B_INIT_FROM_A           = True            # True면 B를 A의 ckpt에서 warm-start (B 자체 ckpt 없을 때만 적용)
B_FINETUNE_LR           = 1e-4            # finetune 시 LR (사전학습 정책 보호용으로 낮춤)
B_FINETUNE_ENT_COEF     = 0.005           # finetune 시 entropy (탐험 줄임)
B_FINETUNE_TOTAL_FRAMES = 500_000         # finetune 은 from-scratch 보다 짧게

# Drive 경로 (A=battery only, B=battery + wind robust)
SAVE_DIR_A   = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_battery"
SAVE_DIR_B   = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_battery_wind" + ("_from_A" if B_INIT_FROM_A else "")
TB_LOGDIR_A  = f"{DRIVE_ROOT}/runs/{EXP_TAG}_battery"
TB_LOGDIR_B  = f"{DRIVE_ROOT}/runs/{EXP_TAG}_battery_wind" + ("_from_A" if B_INIT_FROM_A else "")
GIF_A        = f"{DRIVE_ROOT}/gifs/demo_{EXP_TAG}_battery.gif"
GIF_B        = f"{DRIVE_ROOT}/gifs/demo_{EXP_TAG}_battery_wind.gif"
EVAL_TXT_A   = f"{DRIVE_ROOT}/evals/eval_{EXP_TAG}_battery.txt"
EVAL_TXT_B   = f"{DRIVE_ROOT}/evals/eval_{EXP_TAG}_battery_wind.txt"
SWEEP_PNG    = f"{DRIVE_ROOT}/sweeps/wind_sweep_{EXP_TAG}_battery.png"

print(f"TARGET_SEQUENCE = {TARGET_SEQUENCE}")
print(f"SAVE_DIR_A      = {SAVE_DIR_A}")
print(f"SAVE_DIR_B      = {SAVE_DIR_B}")
print(f"GIF_A / GIF_B   = {GIF_A} | {GIF_B}")
print(f"SWEEP_PNG       = {SWEEP_PNG}")

TARGET_SEQUENCE = GROUND,D,G
SAVE_DIR_A      = /content/drive/MyDrive/drone_results/battery_letters_no_battery/ckpts/DG_battery
SAVE_DIR_B      = /content/drive/MyDrive/drone_results/battery_letters_no_battery/ckpts/DG_battery_wind_from_A
GIF_A / GIF_B   = /content/drive/MyDrive/drone_results/battery_letters_no_battery/gifs/demo_DG_battery.gif | /content/drive/MyDrive/drone_results/battery_letters_no_battery/gifs/demo_DG_battery_wind.gif
SWEEP_PNG       = /content/drive/MyDrive/drone_results/battery_letters_no_battery/sweeps/wind_sweep_DG_battery.png


## 5. Smoke test

환경이 지정한 시퀀스로 reset되는지 + 바람 옵션이 살아있는지 빠르게 확인.

In [8]:
%cd /content/RL-2026s1-tp

from comm_env import BatteryShapeFormationEnv

shapes = [s.strip() for s in TARGET_SEQUENCE.split(',') if s.strip()]

for tag, wp, randw in [("battery (no wind)", 0.0, False),
                        ("battery + wind DR", WIND_PROB, True)]:
    env = BatteryShapeFormationEnv(
        grid_size=GRID_SIZE, n_agents=N_AGENTS, max_steps=MAX_STEPS,
        shapes=shapes,
        initial_battery=INITIAL_BATTERY,
        hover_battery_cost=HOVER_BATTERY_COST,
        move_battery_cost=MOVE_BATTERY_COST,
        low_battery_move_penalty=LOW_BATTERY_MOVE_PENALTY,
        wind_prob=wp, wind_strength=WIND_STRENGTH, randomize_wind=randw,
    )
    obs, _ = env.reset(seed=0)
    print(f"[{tag}] path={env.formation_path.label} | stages={len(env.formation_path.targets)} | obs_dim={env.obs_dim}")

/content/RL-2026s1-tp
[battery (no wind)] path=GROUND->D->G | stages=2 | obs_dim=85
[battery + wind DR] path=GROUND->D->G | stages=2 | obs_dim=85


## 6. Resume helper — Drive에 있는 마지막 ckpt 찾기

셀 8 (정책 A) / 셀 9 (정책 B) 학습이 이 결과를 보고 자동으로 warm-start 여부를 결정.

런타임 끊긴 뒤에는 셀 1~4 재실행 → 이 셀 → 셀 8/9 그대로 재실행이면 끊긴 지점부터 이어짐.

In [9]:
import os, re, glob

def latest_ckpt(save_dir):
    """Return (path, effective_iter) of the truly-latest ckpt in save_dir.

    Scans save_dir itself, plus any legacy `{save_dir}_resume_from_N` folders
    from before this notebook used continuous iter numbering. For ckpts inside
    a `_resume_from_N` folder, the effective iter is `N + filename_iter`,
    so the latest across all folders wins and training can continue with
    incrementing global iter numbers in the main folder.
    """
    cand = []
    # Main folder: effective iter == filename iter
    for p in glob.glob(os.path.join(save_dir, 'ckpt_*.pt')):
        m = re.search(r'ckpt_(\d+)\.pt$', p)
        if m:
            cand.append((int(m.group(1)), p))
    # Legacy resume folders: effective iter = N + filename iter
    for d in glob.glob(f"{save_dir}_resume_from_*"):
        m = re.search(r'_resume_from_(\d+)$', d)
        if not m: continue
        base = int(m.group(1))
        for p in glob.glob(os.path.join(d, 'ckpt_*.pt')):
            mm = re.search(r'ckpt_(\d+)\.pt$', p)
            if mm:
                cand.append((base + int(mm.group(1)), p))
    if not cand:
        return None, 0
    cand.sort()
    return cand[-1][1], cand[-1][0]   # (path, effective_iter)

A_RESUME_CKPT, A_RESUME_ITER = latest_ckpt(SAVE_DIR_A)
B_RESUME_CKPT, B_RESUME_ITER = latest_ckpt(SAVE_DIR_B)

for label, ck, it in [("A (battery)",      A_RESUME_CKPT, A_RESUME_ITER),
                       ("B (battery+wind)", B_RESUME_CKPT, B_RESUME_ITER)]:
    if ck:
        print(f"[{label}] 이어 학습할 ckpt: {ck} (effective iter {it})")
    else:
        print(f"[{label}] ckpt 없음 → 처음부터 학습")

[A (battery)] 이어 학습할 ckpt: /content/drive/MyDrive/drone_results/battery_letters_no_battery/ckpts/DG_battery/ckpt_75.pt (effective iter 75)
[B (battery+wind)] ckpt 없음 → 처음부터 학습


## 7. (선택) 빠른 파이프라인 검증

본 학습 전, GROUND→X 소규모 실행으로 정상 동작만 확인 (수 분). reward 곡선만 확인.

In [10]:
%cd /content/RL-2026s1-tp
!python comm_train_battery.py --shapes GROUND,X --max-steps 250 \
  --total-frames 50000 --frames-per-batch 4096 \
  --wind-prob 0.3 --randomize-wind \
  --ckpt-every 5 --save-dir /tmp/ckpt_smoke_batt_wind --tb-logdir /tmp/runs_smoke_batt_wind

/content/RL-2026s1-tp
2026-05-29 09:50:54.620424: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Battery training: grid_size=25, n_agents=14, max_steps=250, obs_dim=85, shapes='GROUND->X'
Battery: initial=1.0, hover_cost=0.002, move_cost=0.005, low_battery_move_penalty=0.1
Wind: prob=0.3, strength=1, randomize=True
/usr/local/lib/python3.12/dist-packages/torchrl/collectors/_base.py:1188: DeprecationWarning: SyncDataCollector has been deprecated and will be removed in v0.13. Please use Collector instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchrl/collectors/_single.py:906: UserWarning: total_frames (50000) is not exactly divisible by frames_per_batch (4096). This means 3248 additional frames will be collected.To silence this

## 8. 정책 A 학습 — 배터리만 (바람 없음)

Resume 시에도 **같은 폴더** (`SAVE_DIR_A`) 에 계속 저장. 새 ckpt는 마지막 effective iter 다음 번호부터 (예: 마지막이 `ckpt_210.pt` → 다음은 `ckpt_215.pt`, `ckpt_220.pt`, ...).

In [ ]:
%cd /content/RL-2026s1-tp
A_RESUME_CKPT = SAVE_DIR_A + "/ckpt_75.pt"
import os
A_load_arg = ""
A_start_iter = 0
if A_RESUME_CKPT:
    A_load_arg = f"--load-ckpt {A_RESUME_CKPT}"
    A_start_iter = A_RESUME_ITER
    print(f"RESUME [A] from iter {A_start_iter}: {A_RESUME_CKPT}")
    print(f"  -> 같은 폴더에 ckpt_{A_start_iter + CKPT_EVERY}.pt 부터 누적 저장")
else:
    print(f"FRESH START [A] -> {SAVE_DIR_A}")

os.makedirs(SAVE_DIR_A, exist_ok=True)

!python comm_train_battery.py \
  --grid-size {GRID_SIZE} --n-agents {N_AGENTS} --max-steps {MAX_STEPS} \
  --shapes "{TARGET_SEQUENCE}" \
  --completion-reward {COMPLETION_REWARD} \
  --assigned-target-reward {ASSIGNED_TARGET_REWARD} \
  --coverage-delta-reward {COVERAGE_DELTA_REWARD} \
  --hover-penalty {HOVER_PENALTY} --shaping-coef {SHAPING_COEF} \
  --initial-battery {INITIAL_BATTERY} \
  --hover-battery-cost {HOVER_BATTERY_COST} \
  --move-battery-cost {MOVE_BATTERY_COST} \
  --low-battery-move-penalty {LOW_BATTERY_MOVE_PENALTY} \
  --total-frames {TOTAL_FRAMES} \
  --frames-per-batch {FRAMES_PER_BATCH} \
  --minibatch-size {MINIBATCH_SIZE} --ppo-epochs {PPO_EPOCHS} \
  --lr {LR} --ent-coef {ENT_COEF} --clip-eps {CLIP_EPS} \
  --ckpt-every {CKPT_EVERY} \
  --start-iter {A_start_iter} \
  --save-dir {SAVE_DIR_A} \
  --tb-logdir {TB_LOGDIR_A} \
  {A_load_arg}

/content/RL-2026s1-tp
RESUME [A] from iter 75: /content/drive/MyDrive/drone_results/battery_letters_no_battery/ckpts/DG_battery/ckpt_75.pt
  -> 같은 폴더에 ckpt_80.pt 부터 누적 저장
2026-05-29 09:54:30.336138: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Battery training: grid_size=25, n_agents=14, max_steps=500, obs_dim=85, shapes='GROUND->D->G'
Battery: initial=1.0, hover_cost=0.0, move_cost=0.0, low_battery_move_penalty=0.0
Warm-started from /content/drive/MyDrive/drone_results/battery_letters_no_battery/ckpts/DG_battery/ckpt_75.pt
/usr/local/lib/python3.12/dist-packages/torchrl/collectors/_base.py:1188: DeprecationWarning: SyncDataCollector has been deprecated and will be removed in v0.13. Please use Collector instead.
  warnings.warn(
/usr/local/lib

## 9. 정책 B 학습 — 배터리 + 바람 도메인 랜덤화 `[0, WIND_PROB]`

**모드는 자동 결정** (셀 4의 `B_INIT_FROM_A` 토글):
- `B_RESUME_CKPT` 있음 → B 자체 ckpt에서 resume (같은 폴더, iter 번호 누적)
- `B_INIT_FROM_A=True` + A ckpt 있음 → **A의 정책에서 finetune** (LR/entropy/frames는 `B_FINETUNE_*` 값으로 override). 결과는 `..._battery_wind_from_A` 폴더에 저장, iter 카운트는 0부터
- 그 외 → from scratch

Finetune은 from-scratch 대비 학습량을 절반 이하로 줄여도 보통 빨리 수렴함. (A가 이미 포메이션 로직을 알고 있으니 B는 바람 보정만 추가로 학습)

In [ ]:
%cd /content/RL-2026s1-tp

import os
B_lr           = LR
B_ent          = ENT_COEF
B_total_frames = TOTAL_FRAMES
B_load_arg     = ""
B_start_iter   = 0

if B_RESUME_CKPT:
    B_load_arg = f"--load-ckpt {B_RESUME_CKPT}"
    B_start_iter = B_RESUME_ITER
    print(f"RESUME [B] from iter {B_start_iter}: {B_RESUME_CKPT}")
    print(f"  -> 같은 폴더({SAVE_DIR_B}) 에 ckpt_{B_start_iter + CKPT_EVERY}.pt 부터 누적 저장")
elif B_INIT_FROM_A:
    assert A_RESUME_CKPT, "B_INIT_FROM_A=True 인데 A 의 ckpt가 없음. 먼저 셀 8 (정책 A) 학습 필요."
    B_load_arg = f"--load-ckpt {A_RESUME_CKPT}"
    B_lr           = B_FINETUNE_LR
    B_ent          = B_FINETUNE_ENT_COEF
    B_total_frames = B_FINETUNE_TOTAL_FRAMES
    print(f"FINETUNE [B] from A: {A_RESUME_CKPT} -> {SAVE_DIR_B}")
    print(f"  override: lr={B_lr}, ent_coef={B_ent}, total_frames={B_total_frames}")
    print(f"  iter 카운트는 0부터 (A 와 별개 학습)")
else:
    print(f"FRESH START [B] -> {SAVE_DIR_B}")

os.makedirs(SAVE_DIR_B, exist_ok=True)

!python comm_train_battery.py \
  --grid-size {GRID_SIZE} --n-agents {N_AGENTS} --max-steps {MAX_STEPS} \
  --shapes "{TARGET_SEQUENCE}" \
  --completion-reward {COMPLETION_REWARD} \
  --assigned-target-reward {ASSIGNED_TARGET_REWARD} \
  --coverage-delta-reward {COVERAGE_DELTA_REWARD} \
  --hover-penalty {HOVER_PENALTY} --shaping-coef {SHAPING_COEF} \
  --initial-battery {INITIAL_BATTERY} \
  --hover-battery-cost {HOVER_BATTERY_COST} \
  --move-battery-cost {MOVE_BATTERY_COST} \
  --low-battery-move-penalty {LOW_BATTERY_MOVE_PENALTY} \
  --wind-prob {WIND_PROB} --wind-strength {WIND_STRENGTH} --randomize-wind \
  --total-frames {B_total_frames} \
  --frames-per-batch {FRAMES_PER_BATCH} \
  --minibatch-size {MINIBATCH_SIZE} --ppo-epochs {PPO_EPOCHS} \
  --lr {B_lr} --ent-coef {B_ent} --clip-eps {CLIP_EPS} \
  --ckpt-every {CKPT_EVERY} \
  --start-iter {B_start_iter} \
  --save-dir {SAVE_DIR_B} \
  --tb-logdir {TB_LOGDIR_B} \
  {B_load_arg}

## 10. TensorBoard

양쪽 정책의 학습 곡선을 동시에 확인.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_ROOT}/runs

## 11. 평가에 쓸 최신 ckpt 자동 선택

셀 6의 `latest_ckpt`를 다시 호출해서 A/B 각각의 (resume 폴더 포함) 가장 늦은 ckpt를 골라 평가용 변수로 잡음. 원하는 다른 iter 가 있으면 `CKPT_A`/`CKPT_B`를 직접 덮어써도 됨.

In [ ]:
CKPT_A, _ = latest_ckpt(SAVE_DIR_A)
CKPT_B, _ = latest_ckpt(SAVE_DIR_B)
assert CKPT_A, f"No A ckpt found under {SAVE_DIR_A}*"
assert CKPT_B, f"No B ckpt found under {SAVE_DIR_B}*"
print(f"A (battery)      ckpt: {CKPT_A}")
print(f"B (battery+wind) ckpt: {CKPT_B}")

## 12. 정책 A 평가 — 바람 없는 환경에서 GIF + 통계 (Drive 저장)

In [ ]:
%cd /content/RL-2026s1-tp
!python comm_eval_battery.py \
  --ckpt {CKPT_A} \
  --grid-size {GRID_SIZE} --n-agents {N_AGENTS} --max-steps {MAX_STEPS} \
  --shapes "{TARGET_SEQUENCE}" \
  --completion-reward {COMPLETION_REWARD} \
  --initial-battery {INITIAL_BATTERY} \
  --hover-battery-cost {HOVER_BATTERY_COST} \
  --move-battery-cost {MOVE_BATTERY_COST} \
  --low-battery-move-penalty {LOW_BATTERY_MOVE_PENALTY} \
  --greedy --n-episodes 50 \
  --save-gif {GIF_A} \
  --out {EVAL_TXT_A}

print('\n=== eval txt 결과 (A: battery, no wind) ===')
!cat {EVAL_TXT_A}

## 13. 정책 B 평가 — `EVAL_WIND_PROB` 바람 속 GIF + 통계 (Drive 저장)

In [ ]:
%cd /content/RL-2026s1-tp
!python comm_eval_battery.py \
  --ckpt {CKPT_B} \
  --grid-size {GRID_SIZE} --n-agents {N_AGENTS} --max-steps {MAX_STEPS} \
  --shapes "{TARGET_SEQUENCE}" \
  --completion-reward {COMPLETION_REWARD} \
  --initial-battery {INITIAL_BATTERY} \
  --hover-battery-cost {HOVER_BATTERY_COST} \
  --move-battery-cost {MOVE_BATTERY_COST} \
  --low-battery-move-penalty {LOW_BATTERY_MOVE_PENALTY} \
  --wind-prob {EVAL_WIND_PROB} \
  --greedy --n-episodes 50 \
  --save-gif {GIF_B} \
  --out {EVAL_TXT_B}

print('\n=== eval txt 결과 (B: battery + wind) ===')
!cat {EVAL_TXT_B}

## 14. 바람 sweep — A vs B robustness 곡선 PNG

`wind_sweep_battery.py`가 두 정책을 `WIND_LEVELS` 각 점에서 평가하고 비교 곡선 PNG를 Drive에 저장. 곡선이 갈수록 벌어지면 B(바람 DR 학습)가 robust한 것.

In [ ]:
%cd /content/RL-2026s1-tp
!python wind_sweep_battery.py \
  --ckpt-a {CKPT_A} --ckpt-b {CKPT_B} \
  --shapes "{TARGET_SEQUENCE}" --max-steps {MAX_STEPS} \
  --wind-levels {WIND_LEVELS} --n-episodes 100 \
  --completion-reward {COMPLETION_REWARD} \
  --initial-battery {INITIAL_BATTERY} \
  --hover-battery-cost {HOVER_BATTERY_COST} \
  --move-battery-cost {MOVE_BATTERY_COST} \
  --low-battery-move-penalty {LOW_BATTERY_MOVE_PENALTY} \
  --label-a "battery (no wind)" --label-b "battery + wind DR" \
  --out {SWEEP_PNG}

from IPython.display import Image
Image(SWEEP_PNG)

## 15. GIF 표시 (A: 바람 없음 / B: 바람 속)

In [ ]:
from IPython.display import Image, display
print('=== A (battery only, no wind) ===')
display(Image(GIF_A))
print('=== B (battery + wind) ===')
display(Image(GIF_B))

## 16. Drive 산출물 점검

새 세션 복구 시 이 셀로 "지금 Drive에 뭐가 남아있나"만 확인하면 그대로 이어갈 수 있음.

In [ ]:
print("=== 모든 실험 ckpt 폴더 ===")
!ls -la {DRIVE_ROOT}/ckpts/

print(f"\n=== 현재 실험 ({EXP_TAG}) A (battery) ckpt 폴더들 ===")
!for d in {DRIVE_ROOT}/ckpts/{EXP_TAG}_battery*; do echo "  $d:"; ls $d 2>/dev/null | sort -t_ -k2 -n | tail -5; done

print("\n=== sweeps ===")
!ls -la {DRIVE_ROOT}/sweeps/ 2>/dev/null

print("\n=== gifs ===")
!ls -la {DRIVE_ROOT}/gifs/ 2>/dev/null

print("\n=== evals ===")
!ls -la {DRIVE_ROOT}/evals/ 2>/dev/null

---

## 런타임 끊겼을 때 복구 절차

1. 새 GPU 런타임 할당
2. 셀 1 (clone) → 셀 2 (Drive mount) → 셀 3 (pip) → 셀 4 (설정 재확인) 만 다시 실행
3. 셀 6 (resume helper)로 A/B 각각 Drive에 어디까지 학습됐는지 확인
4. 셀 8 (A) / 셀 9 (B) 중 끊긴 쪽 다시 실행 → 자동으로 마지막 ckpt에서 이어 학습
5. 양쪽 학습이 끝나면 셀 11 → 셀 12 → 셀 13 → 셀 14 (sweep)

**중요**: `CKPT_EVERY` 가 작을수록 disconnection 시 손실이 적음 (기본 5 iter).

## 새 시퀀스를 추가로 학습하고 싶을 때

셀 4의 `TARGET_SEQUENCE` 와 `EXP_TAG`를 바꾸고 셀 6 → 8 → 9 → 11 → 12 → 13 → 14 를 다시 실행. `EXP_TAG`가 다르면 별도 폴더에 저장되므로 기존 학습이 덮이지 않음.

## 한쪽 정책만 학습하고 싶을 때

- A(배터리만)만: 셀 8 → (셀 11에서 `CKPT_B = None` 처리 필요) → 셀 12. 셀 14 sweep은 둘 다 있어야 의미가 있어서 skip.
- B(배터리+바람)만: 셀 9 → (셀 11 동일) → 셀 13. sweep 역시 skip.